# **Evaluation LaTeX Table Writer**

This notebook generates LaTeX tables for the evaluation results of the meal planning system. It reads pre-computed metric data from JSON files and produces formatted `longtblr` tables suitable for direct inclusion in the dissertation, useful to avoid having to re-write the tables manually given the evaluation setup (while also ensuring accuracy in the dissertation).

Two types of table are produced:
- Overall results table: summarises the mean and standard deviation of each metric across all six evaluation scenarios.
- Per-scenario tables: individual metric scores for each of the six evaluation scenarios.

These tables are all saved to the `tables/` folder for use in the dissertation.

In [1]:
import json
import os

os.makedirs("./latex", exist_ok=True)

## Environment Setup

The `planners` list defines the display name and JSON key prefix for each planner. The `metrics` list defines the metric key, display name, and optimisation direction (`up` for higher-is-better, `down` for lower-is-better) for each evaluation metric.

In [2]:
planners = [
    ("Random", "RandomMealPlanner"),
    (r"\gls{llm}", "LLMMealPlanner"),
    (r"\gls{ilp}", "ILPMealPlanner"),
    (r"\gls{ga}", "GAMealPlanner"),
]

In [3]:
metrics = [
    ("ingredient_utilisation_score", "Ingredient Utilisation", "up"),
    ("expiry_weighted_utilisation_score", "Expiry-Weighted Utilisation", "up"),
    ("food_waste_score", "Food Waste", "down"),
    ("dietary_constraint_compliance", "Dietary Compliance", "up"),
    ("nutritional_target_score", "Nutritional Target", "up"),
    ("budget_efficiency", "Budget Efficiency", "up"),
    ("pantry_coverage_score", "Pantry Coverage", "up"),
    ("variety_score", "Variety", "up"),
    ("runtime_seconds", "Runtime (seconds)", "down"),
]

In [4]:
def format_value(value):
    return f"{value:.4f}"

In [5]:
def bold(string):
    return rf"\textbf{{{string}}}"

In [6]:
with open("./results/average_planner_metrics.json", "r") as file:
    average_planner_metrics = json.load(file)

## Writing the Tables

### *Overall Results Table*

The overall results table summarises the mean ($\mu$) and standard deviation ($\sigma$) of each metric, averaged across all six evaluation scenarios. The best-performing planner for each metric is highlighted in bold.

In [7]:
body_rows = []

for metric_key, metric_name, direction in metrics:
    dir_sym = r"\uparrow" if direction == "up" else r"\downarrow"

    means = [average_planner_metrics[pk][f"{metric_key}_mean"] for _, pk in planners]
    stds = [average_planner_metrics[pk][f"{metric_key}_std"] for _, pk in planners]

    best = min(means) if direction == "down" else max(means)

    cells = []
    for mean, std in zip(means, stds):
        mean_str = format_value(mean)
        if mean == best:
            mean_str = bold(mean_str)
        cells.append(f"{mean_str} & {format_value(std)}")

    body_rows.append(f"    {metric_name} \\hfill ${dir_sym}$ &\n    " + " &\n    ".join(cells) + r" \\")

# assemble header rows
planner_cells = " & &\n    ".join(rf"\SetCell[c=2]{{c}} {name}" for name, _ in planners)
header1 = r"    \SetCell[r=2]{m,l} Metric &" + "\n    " + planner_cells + r" & \\"
header2 = "    & " + " & ".join([r"$\mu$ & $\sigma$"] * len(planners)) + r" \\"

table = (
    r"""\begin{longtblr}[
    caption = {Mean (\(\mu\)) and standard deviation (\(\sigma\)) of metric scores across all six evaluation scenarios (Source: Author's own work).},
    entry   = {Mean (\(\mu\)) and standard deviation (\(\sigma\)) of metric scores across all six evaluation scenarios.},
    label   = {tab:overall_results},
]{
    width   = \textwidth,
    colspec = {
        X[m,l]
        Q[m,c] Q[m,c]
        Q[m,c] Q[m,c]
        Q[m,c] Q[m,c]
        Q[m,c] Q[m,c]
    },
    hlines,
    hline{3} = {2pt},
    vlines,
    vline{2,4,6,8} = {2pt},
    cells   = {font=\scriptsize},
    row{1,2} = {font=\bfseries\footnotesize},
}
"""
    + header1
    + "\n    \n"
    + header2
    + "\n    \n"
    + "\n    \n".join(body_rows)
    + "\n"
    + r"\end{longtblr}"
)

In [8]:
with open("./latex/average_planner_metrics_table.tex", "w") as file:
    file.write(table)

### *Per-Scenario Tables*

Individual tables are generated for each of the six evaluation scenarios. Since the Random planner is run multiple times per scenario, it reports both a mean and standard deviation; the remaining planners report only a single run value.

In [9]:
with open("./results/planner_metrics.json", "r") as file:
    planner_metrics = json.load(file)

In [10]:
scenario_names = [
    "Standard",
    "Low Budget",
    "Overcrowded Pantry",
    "Dietary Restrictions (Vegan + Gluten-Free)",
    "Dietary Restrictions (Lactose-Free + Vegetarian)",
    "High Nutritional Targets",
]

# (display name, JSON key prefix, has_std)
# Random is run multiple times so has mean and std. Others are single runs
scenario_planners = [
    ("Random", "RandomMealPlanner", True),
    (r"\gls{llm}", "LLMMealPlanner", False),
    (r"\gls{ilp}", "ILPMealPlanner", False),
    (r"\gls{ga}", "GAMealPlanner", False),
]

In [11]:
def build_scenario_table(scenario_num: int, scenario_name: str, data: dict) -> str:
    """
    Builds a LaTeX longtable for a specific scenario's results

    :param scenario_num: scenario number (1-6)
    :type scenario_num: int
    :param scenario_name: descriptive name of the scenario (e.g. "Low Budget")
    :type scenario_name: str
    :param data: dictionary containing all planner metrics, keyed by "PlannerKey|ScenarioNumber" (e.g. "RandomMealPlanner|1")
    :type data: dict

    :return: LaTeX code for the scenario's results table
    :rtype: str
    """
    n = scenario_num

    # header row 1: Metric | \SetCell[c=2] Random & & | LLM | ILP | GA
    h1_parts = [r"    \SetCell[r=2]{m,l} Metric"]
    for name, _, has_std in scenario_planners:
        if has_std:
            h1_parts.append(rf"    \SetCell[c=2]{{c}} {name} & ")
        else:
            h1_parts.append(f"    {name}")
    header1 = " &\n".join(h1_parts) + r" \\"

    # header row 2: blank | mu | sigma | blank | blank | blank
    h2_cols = [""]  # blank for metric column
    for _, _, has_std in scenario_planners:
        if has_std:
            h2_cols += [r"$\mu$", r"$\sigma$"]
        else:
            h2_cols.append("")
    header2 = "    " + " & ".join(h2_cols) + r" \\"

    # body rows (actual evaluation values)
    body_rows = []
    for metric_key, metric_name, direction in metrics:
        dir_sym = r"\uparrow" if direction == "up" else r"\downarrow"

        means = [data[f"{planner_key}|{n}"][f"{metric_key}_mean"] for _, planner_key, _ in scenario_planners]
        stds = [data[f"{planner_key}|{n}"][f"{metric_key}_std"] for _, planner_key, _ in scenario_planners]

        best = min(means) if direction == "down" else max(means)

        column_values = []
        for i, (_, _, has_std) in enumerate(scenario_planners):
            mean_str = format_value(means[i])
            if means[i] == best:
                mean_str = bold(mean_str)
            if has_std:
                column_values.append(f"{mean_str} & {format_value(stds[i])}")
            else:
                column_values.append(mean_str)

        body_rows.append(f"    {metric_name} \\hfill ${dir_sym}$ &\n    " + " &\n    ".join(column_values) + r" \\")

    colspec_lines = "        X[m,l]\n        Q[m,c] Q[m,c]\n" + "\n".join(
        "        Q[m,c]" for _, _, has_std in scenario_planners if not has_std
    )

    table = (
        "\\begin{longtblr}[\n"
        f"    caption = {{Metric scores for Scenario {n}: {scenario_name} (Source: Author's own work).}},\n"
        f"    entry   = {{Metric scores for Scenario {n}: {scenario_name}.}},\n"
        f"    label   = {{tab:scenario{n}_results}},\n"
        "]{\n"
        "    width   = \\textwidth,\n"
        "    colspec = {\n"
        f"{colspec_lines}\n"
        "    },\n"
        "    hlines,\n"
        "    hline{3} = {2pt},\n"
        "    vlines,\n"
        "    vline{2,4,6,8} = {2pt},\n"
        "    cells   = {font=\\scriptsize},\n"
        "    row{1,2} = {font=\\bfseries\\footnotesize},\n"
        "}\n" + header1 + "\n    \n" + header2 + "\n    \n" + "\n    \n".join(body_rows) + "\n" + "\\end{longtblr}"
    )
    return table

In [12]:
for i, name in enumerate(scenario_names, start=1):
    scenario_table = build_scenario_table(i, name, planner_metrics)
    filename = f"./latex/scenario{i}_planner_metrics_table.tex"

    with open(filename, "w") as file:
        file.write(scenario_table)